In [0]:
import subprocess
subprocess.check_call(['pip', 'install', 'faker', 'holidays'])

#from utils import save_to_parquet
import numpy as np, pandas as pd, random
from faker import Faker
import holidays

np.random.seed(42)
random.seed(42)
fake = Faker()
Faker.seed(42)

# Get catalog and schema from notebook widgets / job parameters
# Widgets are automatically created from base_parameters in the workflow
# For local or ad‑hoc runs, defaults are used
dbutils.widgets.text("CATALOG_NAME", "your_workspace_catalog", "Catalog")
dbutils.widgets.text("SCHEMA_NAME", "glucosphere", "Schema")

try:
    CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
    SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
except Exception as e:
    raise RuntimeError(f"Widget lookup failed, catalog/schema not passed to notebook: {e}")

print(f"Using catalog: {CATALOG_NAME}, schema: {SCHEMA_NAME}")

# Ensure Unity Catalog structure exists for data storage
print("Setting up Unity Catalog structure...")
try:
    # Ensure catalog exists (if you don't have permission to create catalogs,
    # comment this out and create the catalog once via SQL instead)
    #spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
    print(f"✓ Catalog '{CATALOG_NAME}' ready")
    
    # Ensure schema exists
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
    print(f"✓ Schema '{SCHEMA_NAME}' ready")
    
    # NOW set current catalog and schema (after creating them)
    spark.sql(f"USE CATALOG {CATALOG_NAME}")
    spark.sql(f"USE SCHEMA {SCHEMA_NAME}")
    
    # Ensure volume exists
    spark.sql(f"""
        CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.landing_zone
        COMMENT 'Landing zone for MedTech raw data files'
    """)
    print("✓ Volume 'landing_zone' ready")
    
    # Create subdirectories for all datasets using dbutils
    volume_base = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/landing_zone"
    datasets = [
        "raw_patient_registry",
        "raw_device_telemetry_stream"
    ]
    
    for dataset in datasets:
        dataset_path = f"{volume_base}/{dataset}"
        try:
            dbutils.fs.mkdirs(dataset_path)
            print(f"✓ Volume directory '{dataset}' created")
        except Exception as dir_error:
            print(f"⚠ Could not create directory {dataset}: {dir_error}")
    
except Exception as e:
    print(f"⚠ Could not create Unity Catalog structure: {e}")
    print("  You may need to create catalog/schema/volume manually before running this script")


In [0]:
# Device & patient settings
PATIENT_START = pd.Timestamp("2022-06-01").floor("ms")
PATIENT_END   = pd.Timestamp("2025-11-16").floor("ms")
REGIONS = ["NA", "EMEA", "APAC"]
REGION_PROBS = np.array([0.45, 0.32, 0.23])
DEVICE_MODELS = ["Alpha", "Beta", "Gamma", "Delta", "Epsilon", "Zeta"]  # Older: Alpha/Beta
DIAGNOSES = ["T1D", "T2D", "gestational"]
DIAG_PROBS = np.array([0.45, 0.48, 0.07])
#FIRMWARE_BASE = "v3.14"
#FIRMWARE_EVENT = "v4.00"
#FIRMWARE_FIX = "v4.10"
COVERAGE_TYPES = ["insurance", "public", "self-pay"]
COVERAGE_PROBS = np.array([0.68, 0.22, 0.10])
CARE_EPISODE_TYPES = ["intervention", "routine visit", "telehealth", "acute admission", "icu"]
CARE_EP_TYPE_PROBS = np.array([0.39, 0.34, 0.18, 0.07, 0.02])


In [0]:
# ========== Patient Registry ==========
def generate_patient_registry(n_patients = 1000):
    device_df = spark.table(f"{CATALOG_NAME}.{SCHEMA_NAME}.patient_device")
    print(f"Generating patient_registry ({n_patients:,}) ...")
    device_ids = device_df.select("device_id").distinct().toPandas()["device_id"].tolist()
    patient_ids = device_df.select("patient_id").distinct().toPandas()["patient_id"].tolist()
    # Distribution: T1D ~45%, T2D ~48%, gestational ~7%
    diagnosis = np.random.choice(DIAGNOSES, size=n_patients, p=DIAG_PROBS)
    region = np.random.choice(REGIONS, size=n_patients, p=REGION_PROBS)
    device_model = np.random.choice(DEVICE_MODELS, size=n_patients,
                                  p=[0.22, 0.22, 0.19, 0.16, 0.13, 0.08])
    sample_birth_years = np.concatenate([
            np.random.randint(1940, 1958, size=int(n_patients*0.18)),   # 65+
            np.random.randint(1958, 1984, size=int(n_patients*0.43)),   # 41-64
            np.random.randint(1984, 2007, size=int(n_patients*0.32)),   # 18-40
            np.random.randint(2007, 2022, size=int(n_patients*0.07)),   # <18
        ])
    if len(sample_birth_years) < n_patients:
        sample_birth_years = np.resize(sample_birth_years, n_patients)
    birth_year = np.random.choice(sample_birth_years, size=n_patients, replace=True)

    # Use device_id and patient_id from table
    device_id = device_ids
    patient_id = patient_ids
    # Activation dates are spread
    activation_base = pd.date_range(PATIENT_START - pd.Timedelta(days=90), PATIENT_START, freq="D")
    activation_date = np.random.choice(activation_base, size=n_patients)
    df = pd.DataFrame({
        "patient_id": patient_id,
        "device_id": device_id,
        "region": region,
        "diagnosis": diagnosis,
        "activation_date": pd.to_datetime(activation_date).normalize(),
        "birth_year": birth_year,
        "device_model": device_model
    })
    print("Patient registry generated.")
    return df

In [0]:
from pyspark.sql.functions import count_distinct, collect_list

raw_data_name = "raw_patient_registry"
path = f"{volume_base}/{raw_data_name}"  

device_df = spark.table(f"{CATALOG_NAME}.{SCHEMA_NAME}.patient_device")
n_patients = device_df.agg(count_distinct("patient_id")).collect()[0][0]

patient_registry = generate_patient_registry(n_patients=n_patients)
spark.createDataFrame(patient_registry).write.mode("overwrite").parquet(path)
display(patient_registry)